## DNN

### Data preprocessing

In [ ]:
import pandas as pd

In [ ]:
merged_df=pd.read_csv("/content/Dataset.csv")

In [ ]:
merged_df.head()

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Create a LabelEncoder instance
label_encoder = LabelEncoder()

columns = ['Flow ID','Src IP','Dst IP','Timestamp']

# Apply Label Encoding
for column in columns:
    merged_df[column] = label_encoder.fit_transform(merged_df[column])

merged_df.head()

In [ ]:
features = merged_df.drop('target', axis=1)
labels = merged_df['target']

### Importing necessary liberaries

In [ ]:
!pip install tensorflow

In [ ]:
# Import necessary libraries
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

### Loading and splitting the dataset

In [ ]:
x,y = features,labels # This is a placeholder

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [ ]:
# Build the DNN model
model = Sequential()

# Input layer (83 input features)
model.add(Dense(16, input_dim=83, activation='relu'))

# Hidden layers
model.add(Dense(256, activation='relu'))
model.add(Dropout(0.1))  # Dropout to prevent overfitting
model.add(Dense(128, activation='relu'))

# Output layer (binary classification)
model.add(Dense(1, activation='sigmoid'))  # Sigmoid for binary classification

# Compile the model
model.compile(optimizer=Adam(learning_rate=0.0001),
              loss='binary_crossentropy',
              metrics=['accuracy'])

### Training the Model

In [ ]:
# Train the model
history = model.fit(X_train, y_train,epochs=30,batch_size=32,validation_data=(X_test, y_test))

In [ ]:
# Evaluate the model on the test data
test_loss, test_acc = model.evaluate(X_test, y_test)
print('Test accuracy:', test_acc)

# Predict the test set results and generate the classification report
y_pred = (model.predict(X_test) > 0.5).astype("int32")  # Convert probabilities to binary labels

# Generate a classification report
print(classification_report(y_test, y_pred))

Testing accuracy

In [ ]:
# Evaluate the model on the test data
test_loss, test_acc = model.evaluate(X_test, y_test)
print('Test accuracy:', test_acc)

# Predict the test set results and generate the classification report
y_pred = (model.predict(X_test) > 0.5).astype("int32")  # Convert probabilities to binary labels

# Generate a classification report
print(classification_report(y_test, y_pred))

## Graphs and plotting

In [ ]:
!pip install scikit-plot

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc

In [ ]:
from sklearn.metrics import classification_report

# Get classification report
report = classification_report(y_test, y_pred, output_dict=True)
report_df = pd.DataFrame(report).transpose()  # Convert to DataFrame for easy plotting

In [ ]:
# Generate the confusion matrix
conf_matrix = confusion_matrix(y_test, y_pred)

# Plot the confusion matrix
plt.figure(figsize=(6,4))
sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.title("Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()

In [ ]:
# Extract loss and validation loss
train_loss = history.history['loss']
val_loss = history.history['val_loss']

# Plotting training and validation loss
plt.figure(figsize=(8,6))
plt.plot(train_loss, label='Training Loss')
plt.plot(val_loss, label='Validation Loss', linestyle='--')
plt.title('Training and Validation Loss per Epoch')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

## unwanted

In [ ]:
# Plot Precision, Recall, F1-score for each class
metrics = ['precision', 'recall', 'f1-score']

# Plot a bar graph for precision, recall, and f1-score
report_df = report_df.loc[['0', '1'], metrics]  # Filter out support and accuracy
report_df.plot(kind="bar", figsize=(8,6))
plt.title("Precision, Recall, and F1-Score for each class")
plt.ylabel("Score")
plt.xticks(rotation=0)
plt.show()

In [ ]:
from sklearn.metrics import roc_curve, auc

# Compute ROC curve and ROC area for each class
fpr, tpr, _ = roc_curve(y_test, y_pred)
roc_auc = auc(fpr, tpr)

# Plot ROC curve
plt.figure(figsize=(8,6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc="lower right")
plt.show()

In [ ]:
# Regenerate the classification report with 'output_dict=True'
report = classification_report(y_test, y_pred, output_dict=True)
report_df = pd.DataFrame(report).transpose()

print(report_df)


In [ ]:
# Plot the accuracy and support for each class
report_df_acc_sup = report_df.loc[['0', '1'], ['support']]
report_df_acc_sup.plot(kind="bar", figsize=(8,6), color='green')
plt.title("Support for each class")
plt.ylabel("Count")
plt.xticks(rotation=0)
plt.show()

##Model testing


In [ ]:
#Leave one out
import numpy as np
from sklearn.model_selection import train_test_split

def evaluate_trained_model(X, y, model, test_size=0.2, random_state=42):
    """
    Evaluates a pre-trained neural network model using holdout validation.

    Parameters:
    X (numpy.ndarray): Input features
    y (numpy.ndarray): Target variable
    model (tensorflow.keras.models.Sequential): Trained neural network model
    test_size (float, optional): Proportion of data to use for the test set (default is 0.2)
    random_state (int, optional): Random state for reproducibility (default is 42)

    Returns:
    tuple: (train_score, test_score)
        train_score (float): Model performance on the training set
        test_score (float): Model performance on the test set
    """
    # Split the data into train and test sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state)

    # Evaluate the model on the training and test sets
    train_score = model.evaluate(X_train, y_train, verbose=0)[1]
    test_score = model.evaluate(X_test, y_test, verbose=0)[1]

    return train_score, test_score

In [ ]:
evaluate_trained_model(x,y,model)

##  Pushing the model to hugging face

In [ ]:
# Save the model using TensorFlow's SavedModel format
model.save('saved_model.keras')

In [ ]:
from huggingface_hub import Repository

# Clone the Hugging Face repository
repo = Repository(local_dir="ddos_detection_repo", clone_from="Sharukesh/ddos-detection-model")

# Move the .keras file to the repository directory
import shutil
shutil.move('/content/saved_model.keras', 'ddos_detection_repo/ddos_detection_model.keras')

# Push the model to Hugging Face
repo.push_to_hub(commit_message="Initial commit of DDoS detection model in .keras format")


In [ ]:
print(merged_df.columns)

In [ ]:
from keras.models import load_model

# Load the model from a .keras file
model = load_model('/content/dnn_model.keras')

# Check model summary to ensure it's loaded correctly
model.summary()